# VaR_Cal — 포트폴리오 리스크 분석 노트북
## `VaR_Cal.ipynb`

| 항목 | 내용 |
|------|------|
| Parametric VaR | EWMA(λ=0.94) + Marginal / Component / Incremental VaR |
| Historical VaR | 과거 수익률 분위수 직접 추출 (분포 가정 없음) |
| Monte Carlo VaR | EWMA 공분산 + t-분포 다변량 시뮬레이션 |
| 시뮬레이션 | Merton Jump Diffusion / Block Bootstrap / t-MC 경로 분석 |
| Back-testing | 역사적 손익 vs VaR, Kupiec POF 검정, Basel 신호등 |
| 포트폴리오 업로드 | Excel / CSV 파일로 포지션 일괄 입력 |
| 리포트 | Markdown(.md) + Excel(.xlsx) 자동 생성 |

> **Streamlit 앱 실행**: `streamlit run VaR_Cal.py`

## 0. 패키지 설치 및 임포트

In [1]:
# !pip install yfinance plotly scipy pandas numpy openpyxl

import numpy as np
import pandas as pd
from scipy import stats, interpolate
from datetime import datetime, timedelta
import warnings; warnings.filterwarnings("ignore")

try:
    import yfinance as yf; YF_OK=True; print("yfinance 연결됨")
except ImportError:
    YF_OK=False; print("yfinance 미설치 — 샘플 데이터 사용")

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots; PLOTLY=True
except ImportError:
    import matplotlib.pyplot as plt; PLOTLY=False; print("plotly 없음 — matplotlib 사용")

Z99, Z95 = 2.3263, 1.6449
TENORS   = np.array([0.25,0.5,1,2,3,5,7,10,20,30])
COLORS   = ["#185FA5","#A32D2D","#3B6D11","#854F0B","#534AB7","#0F6E56"]

ModuleNotFoundError: No module named 'scipy'

## 1. 포트폴리오 정의

포지션을 직접 입력하거나, Excel/CSV 파일로 불러올 수 있습니다.

### 파일 업로드 형식 (필수 컬럼)
`ticker, type, qty, price`

### 선택 컬럼
`ccy, market, coupon, maturity, cf_freq, face_value, strike, callput, iv, mat, rf, mult`

In [ ]:
# ── 옵션 A: 파일에서 불러오기 ─────────────────────────────────────────
def load_portfolio_file(filepath):
    """Excel/CSV 파일로부터 포지션 목록 파싱"""
    if filepath.endswith(".csv"):
        df = pd.read_csv(filepath)
    else:
        df = pd.read_excel(filepath)
    df.columns = [c.strip().lower().replace(" ","_") for c in df.columns]
    if "ccy" not in df.columns:    df["ccy"] = "KRW"
    if "market" not in df.columns:
        def guess(row):
            tk=str(row.get("ticker","")).upper(); t=str(row.get("type","")).capitalize()
            if t=="Bond": return "KR-BOND"
            if t in ("Option","Future"): return "KR-DERIV"
            if tk.endswith(".KS"): return "KR-KOSPI"
            if tk.endswith(".KQ"): return "KR-KOSDAQ"
            return "US-NASDAQ"
        df["market"]=df.apply(guess,axis=1)
    positions=[]
    for _,row in df.iterrows():
        try:
            pos={"ticker":str(row["ticker"]).strip().upper(),"type":str(row["type"]).capitalize(),
                 "qty":float(row["qty"]),"price":float(row["price"]),
                 "ccy":str(row.get("ccy","KRW")).upper(),"market":str(row.get("market","KR-KOSPI")),
                 "mult":float(row["mult"]) if "mult" in row and pd.notna(row.get("mult")) else 1.0}
            for f in ["coupon","maturity","cf_freq","face_value","strike","callput","iv","mat","rf"]:
                if f in row and pd.notna(row.get(f)): pos[f]=row[f]
            positions.append(pos)
        except: continue
    return positions

# ── 옵션 B: 직접 입력 ─────────────────────────────────────────────────
POSITIONS = [
    {"ticker":"005930.KS","type":"Stock","qty":100,"price":75000,
     "ccy":"KRW","market":"KR-KOSPI","mult":1},
    {"ticker":"KR103501GA96","type":"Bond","qty":10000,"price":10000,
     "ccy":"KRW","market":"KR-BOND","coupon":0.035,"maturity":3.0,"cf_freq":2,"face_value":10000,"mult":1},
    {"ticker":"K200C-255-202506","type":"Option","qty":10,"price":257.0,
     "ccy":"KRW","market":"KR-DERIV","strike":255,"callput":"C","iv":0.18,"mat":0.16,"rf":0.035,"mult":250000},
    {"ticker":"AAPL","type":"Stock","qty":50,"price":185,
     "ccy":"USD","market":"US-NASDAQ","mult":1},
]

# 파일에서 불러오려면:
# POSITIONS = load_portfolio_file("my_portfolio.xlsx")

print(f"포지션 수: {len(POSITIONS)}")
for p in POSITIONS:
    print(f"  {p['ticker']:25s} {p['type']:8s} qty={p['qty']:,.0f}  price={p['price']:,.0f}  {p['ccy']}")

## 2. 실시간 환율 및 포트폴리오 현재가 일괄 조회

In [ ]:
USDKRW = 1370   # 기본값 (아래에서 실시간 갱신)

def get_usdkrw():
    if not YF_OK: return 1370.0, False
    try:
        p=yf.Ticker("USDKRW=X").fast_info.last_price
        return (round(float(p),2),True) if p and p>0 else (1370.0,False)
    except: return 1370.0,False

def fetch_prices_bulk(tickers):
    """복수 티커 현재가 yfinance 1회 호출로 일괄 조회"""
    result={}
    if not YF_OK or not tickers: return result
    try:
        raw=yf.download(list(tickers),period="2d",progress=False,auto_adjust=True,
                        group_by="ticker" if len(tickers)>1 else "column")
        for tk in tickers:
            try:
                col=raw[tk]["Close"] if len(tickers)>1 else raw["Close"]
                v=float(col.dropna().iloc[-1])
                if v>0: result[tk]=v
            except: pass
    except: pass
    for tk in tickers:
        if tk not in result and YF_OK:
            try:
                p=yf.Ticker(tk).fast_info.last_price
                if p and p>0: result[tk]=float(p)
            except: pass
    return result

# 환율 조회
USDKRW, fx_live = get_usdkrw()
print(f"USD/KRW: {USDKRW:,.2f} ({'실시간' if fx_live else '기본값'})")

# 주식 현재가 일괄 조회
stock_tickers = tuple(p["ticker"] for p in POSITIONS if p["type"]=="Stock")
live_px = fetch_prices_bulk(stock_tickers)
for p in POSITIONS:
    if p["type"]=="Stock" and p["ticker"] in live_px:
        old=p["price"]; p["price"]=live_px[p["ticker"]]
        print(f"  {p['ticker']}: {old:,.0f} → {p['price']:,.0f} ({'실시간' if fx_live else '기본값'})")

# 총 평가금액
total_mv = sum(p["price"]*p["qty"]*(USDKRW if p["ccy"]=="USD" else 1)*p.get("mult",1)
               for p in POSITIONS)
print(f"\n총 평가금액: {total_mv/1e8:.2f}억원")

## 3. 수익률 데이터 수집 및 기초 통계량

In [ ]:
OBS_DAYS = 504   # 2년

def fetch_returns(ticker, days=504):
    if YF_OK:
        try:
            end=datetime.today(); start=end-timedelta(days=int(days*1.6))
            d=yf.download(ticker,start=start.strftime("%Y-%m-%d"),
                          end=end.strftime("%Y-%m-%d"),progress=False,auto_adjust=True)
            if not d.empty and len(d)>=60:
                px=d["Close"].squeeze(); ret=np.log(px/px.shift(1)).dropna().values
                return ret[-days:], float(px.iloc[-1]), True
        except: pass
    rng=np.random.default_rng(abs(hash(ticker))%9999); sigma=0.22/np.sqrt(252)
    return rng.normal(-0.5*sigma**2,sigma,days), 50000.0, False

def fetch_returns_multi(tickers, days=504):
    ret_dict={}; price_dict={}; any_live=False
    for tk in tickers:
        r,px,live=fetch_returns(tk,days)
        ret_dict[tk]=r; price_dict[tk]=px; any_live=any_live or live
    return ret_dict, price_dict, any_live

print("수익률 수집 중...")
tickers=[p["ticker"] for p in POSITIONS]
RETURNS,_,live_ret=fetch_returns_multi(tickers,OBS_DAYS)
for tk in tickers:
    r=RETURNS[tk]
    print(f"  {tk}: {len(r)}일 | σ연율={np.std(r)*np.sqrt(252)*100:.1f}% | "
          f"μ연율={np.mean(r)*252*100:.2f}%")

# 기초 통계량
from scipy.stats import skew as _skew, kurtosis as _kurt, jarque_bera as _jb
rows=[]
for tk in tickers:
    r=RETURNS[tk]; jb_s,jb_p=_jb(r)
    rows.append({"티커":tk,"연율수익률(%)":round(np.mean(r)*252*100,2),
                 "연율변동성(%)":round(np.std(r)*np.sqrt(252)*100,2),
                 "왜도":round(float(_skew(r)),3),"초과첨도":round(float(_kurt(r)),3),
                 "JB p":round(float(jb_p),4),"정규분포":"기각" if jb_p<0.05 else "채택"})
print("\n=== 기초 통계량 ===")
print(pd.DataFrame(rows).to_string(index=False))

## 4. Parametric VaR + Historical Simulation VaR

### 이론적 배경
| 방법 | 수식 | 특징 |
|------|------|------|
| **Parametric (EWMA)** | $\text{VaR} = z_\alpha \cdot \sigma_{EWMA} \cdot \sqrt{h} \cdot MV$ | 정규분포 가정, 계산 효율 높음 |
| **Historical Simulation** | $\text{VaR} = -Q_{\alpha}(r_t \cdot \sqrt{h})$ | 분포 가정 없음, Fat-tail 자동 반영 |

- **Marginal VaR**: $\partial \text{VaR}_p / \partial w_i$
- **Component VaR**: $w_i \times \text{MVaR}_i \times MV_p$  
- **Incremental VaR**: $\text{VaR}_p - \text{VaR}_{p \setminus i}$

In [ ]:
CONF    = 0.99
HOLDING = 10      # 영업일 (바젤 기준)
LAM     = 0.94    # EWMA 감쇄계수

def ewma_vol(returns, lam=0.94):
    n=len(returns); v=np.zeros(n); v[0]=returns[0]**2
    for t in range(1,n): v[t]=lam*v[t-1]+(1-lam)*returns[t-1]**2
    return np.sqrt(v)

def ewma_cov_mat(ret_matrix, lam=0.94):
    T,n=ret_matrix.shape
    w=np.array([(1-lam)*lam**(T-1-t) for t in range(T)]); w/=w.sum()
    mu=ret_matrix.T@w; d=ret_matrix-mu
    return (d*w[:,None]).T@d

def param_var(ret, mv, holding=10, conf=0.99, lam=0.94):
    sig=ewma_vol(ret,lam)[-1]; z=stats.norm.ppf(conf)
    var=z*sig*np.sqrt(holding)*mv
    cvar=(stats.norm.pdf(z)/(1-conf))*sig*np.sqrt(holding)*mv
    return {"sig_d":sig,"var":var,"cvar":cvar,"var_pct":var/mv}

def hist_var(ret, mv, holding=10, conf=0.99):
    """Historical Simulation VaR"""
    rets_h=ret*np.sqrt(holding); alpha=1-conf
    vp=float(-np.percentile(rets_h,alpha*100))
    cm=rets_h<=-vp; cp=float(-rets_h[cm].mean()) if cm.any() else vp
    return {"var":vp*mv,"cvar":cp*mv,"var_pct":vp}

def pf_var_ewma(weights, cov_mat, pf_mv, conf=0.99, holding=10):
    w=np.array(weights); pv=float(w@cov_mat@w)
    sig_p=np.sqrt(max(pv,0)); z=stats.norm.ppf(conf)
    pf_var=z*sig_p*np.sqrt(holding)*pf_mv
    mvar=(z*np.sqrt(holding)*cov_mat@w/sig_p if sig_p>0 else np.zeros(len(w)))
    comp=w*mvar*pf_mv; beta=cov_mat@w/pv if pv>0 else np.zeros(len(w))
    return {"pf_var":pf_var,"pf_sigma":sig_p,"mvar":mvar,"comp":comp,"beta":beta}

def hist_pf_var(ret_dict, tickers, weights, total_mv, holding=10, conf=0.99):
    """포트폴리오 Historical VaR"""
    ml=min(len(ret_dict[tk]) for tk in tickers)
    pf=np.zeros(ml)
    for tk,w in zip(tickers,weights): pf+=ret_dict[tk][-ml:]*w
    rets_h=pf*np.sqrt(holding); alpha=1-conf
    vp=float(-np.percentile(rets_h,alpha*100))
    cm=rets_h<=-vp; cp=float(-rets_h[cm].mean()) if cm.any() else vp
    return {"pf_var":vp*total_mv,"pf_cvar":cp*total_mv,"var_pct":vp}

def incr_var(weights, cov_mat, pf_mv, conf=0.99, holding=10):
    base=pf_var_ewma(weights,cov_mat,pf_mv,conf,holding)["pf_var"]
    w=np.array(weights); ivars=[]
    for i in range(len(w)):
        mask=[j for j in range(len(w)) if j!=i]
        if not mask: ivars.append(base); continue
        we=w[mask]/w[mask].sum(); ce=cov_mat[np.ix_(mask,mask)]
        mve=pf_mv*w[mask].sum()
        ivars.append(base-pf_var_ewma(we,ce,mve,conf,holding)["pf_var"])
    return ivars

# ── 실행 ─────────────────────────────────────────────────────────────────
mv_list  = [p["price"]*p["qty"]*(USDKRW if p["ccy"]=="USD" else 1)*p.get("mult",1)
            for p in POSITIONS]
total_mv = sum(mv_list)
weights  = [m/total_mv for m in mv_list]

print(f"=== 개별 종목 Parametric vs Historical VaR ({int(CONF*100)}%, {HOLDING}일) ===")
ind_pvars=[]; ind_hvars=[]; rows_vh=[]
for p,mv in zip(POSITIONS,mv_list):
    tk=p["ticker"]; r=RETURNS[tk]
    pv=param_var(r,mv,HOLDING,CONF,LAM)
    hv=hist_var(r,mv,HOLDING,CONF)
    ind_pvars.append(pv["var"]); ind_hvars.append(hv["var"])
    diff=(pv["var"]-hv["var"])/hv["var"]*100 if hv["var"] else 0
    rows_vh.append({"티커":tk,"평가금액(억)":round(mv/1e8,3),
                    "P-VaR":f"{pv['var']/1e6:.1f}M","P-VaR%":f"{pv['var_pct']*100:.3f}%",
                    "H-VaR":f"{hv['var']/1e6:.1f}M","H-VaR%":f"{hv['var_pct']*100:.3f}%",
                    "P vs H":f"{diff:+.1f}%"})
print(pd.DataFrame(rows_vh).to_string(index=False))

min_len = min(len(RETURNS[tk]) for tk in tickers)
ret_mat = np.column_stack([RETURNS[tk][-min_len:] for tk in tickers])
cov_mat = ewma_cov_mat(ret_mat, LAM)
pf_res  = pf_var_ewma(weights, cov_mat, total_mv, CONF, HOLDING)
h_res   = hist_pf_var(RETURNS, tickers, weights, total_mv, HOLDING, CONF)
ivars   = incr_var(weights, cov_mat, total_mv, CONF, HOLDING)

undiv_p = sum(ind_pvars); div_eff = undiv_p - pf_res["pf_var"]
print(f"\n=== 포트폴리오 VaR ({int(CONF*100)}%, {HOLDING}일) ===")
print(f"Parametric VaR : {pf_res['pf_var']/1e6:.1f}M원")
print(f"Historical VaR : {h_res['pf_var']/1e6:.1f}M원")
print(f"분산 효과      : {div_eff/1e6:.1f}M원 ({div_eff/undiv_p*100:.1f}%)")
print(f"포트폴리오 σ({HOLDING}일): {pf_res['pf_sigma']*100:.4f}%")

print(f"\n=== Component / Marginal / Incremental VaR ===")
comp_rows=[]
for i,tk in enumerate(tickers):
    comp=pf_res["comp"][i]; contrib=comp/pf_res["pf_var"]*100 if pf_res["pf_var"] else 0
    comp_rows.append({"티커":tk,"비중":f"{weights[i]*100:.1f}%",
                      "P-VaR(M)":f"{ind_pvars[i]/1e6:.2f}","H-VaR(M)":f"{ind_hvars[i]/1e6:.2f}",
                      "Marginal":f"{pf_res['mvar'][i]:.4f}","Component(M)":f"{comp/1e6:.2f}",
                      "기여도":f"{contrib:.1f}%","IVaR(M)":f"{ivars[i]/1e6:.2f}",
                      "Beta":f"{pf_res['beta'][i]:.3f}"})
print(pd.DataFrame(comp_rows).to_string(index=False))

### 4-1. Parametric vs Historical VaR 비교 차트

In [ ]:
if PLOTLY:
    fig=go.Figure()
    fig.add_trace(go.Bar(x=tickers,y=[v/1e6 for v in ind_pvars],name="Parametric VaR",
        marker_color="#A32D2D",text=[f"{v/1e6:.1f}M" for v in ind_pvars],textposition="outside"))
    fig.add_trace(go.Bar(x=tickers,y=[v/1e6 for v in ind_hvars],name="Historical VaR",
        marker_color="#854F0B",text=[f"{v/1e6:.1f}M" for v in ind_hvars],textposition="outside"))
    fig.update_layout(title=f"종목별 P-VaR vs H-VaR ({int(CONF*100)}%, {HOLDING}일, 백만원)",
                      barmode="group",height=340,legend=dict(orientation="h",y=-0.2))
    fig.show()
else:
    x=np.arange(len(tickers)); w=0.35
    plt.figure(figsize=(10,4))
    plt.bar(x-w/2,[v/1e6 for v in ind_pvars],w,label="Parametric",color="#A32D2D")
    plt.bar(x+w/2,[v/1e6 for v in ind_hvars],w,label="Historical",color="#854F0B")
    plt.xticks(x,tickers); plt.ylabel("VaR (백만원)")
    plt.title("Parametric vs Historical VaR"); plt.legend(); plt.tight_layout(); plt.show()

## 5. Monte Carlo Simulation VaR

EWMA 공분산 행렬 + t-분포 다변량 시뮬레이션으로 VaR를 계산합니다.

$$r_t^{MC} = \mu \cdot h + L \cdot \sqrt{\frac{\nu-2}{\nu}} \cdot Z_\nu, \quad Z_\nu \sim t_\nu$$

- $L$: EWMA 공분산 행렬의 Cholesky 분해 ($\Sigma_h = LL^T$)
- $Z_\nu$: 자유도 $\nu$의 다변량 t-분포 잡음

In [ ]:
MC_N    = 10000   # 시뮬레이션 횟수
MC_DF   = 5       # t-분포 자유도
MC_SEED = 42
USE_T   = True    # False 시 정규분포 사용

def mc_var_individual(ret, mv, holding=10, conf=0.99,
                       n=10000, use_t=True, df=5, lam=0.94, seed=42):
    """개별 종목 Monte Carlo VaR"""
    rng=np.random.default_rng(seed)
    sig=ewma_vol(ret,lam)[-1]; mu_d=float(np.mean(ret))
    if use_t and df>2:
        scale=sig*np.sqrt((df-2)/df)
        z=stats.t.rvs(df=df,size=n,random_state=int(rng.integers(0,2**31)))
        sim=mu_d*holding+scale*np.sqrt(holding)*z
    else:
        sim=rng.normal(mu_d*holding,sig*np.sqrt(holding),n)
    alpha=1-conf; vp=float(-np.percentile(sim,alpha*100))
    cm=sim<=-vp; cp=float(-sim[cm].mean()) if cm.any() else vp
    return {"var":vp*mv,"cvar":cp*mv,"var_pct":vp,"sim":sim}

def mc_var_portfolio(ret_dict, tickers, weights, total_mv,
                      holding=10, conf=0.99,
                      n=10000, use_t=True, df=5, lam=0.94, seed=42):
    """포트폴리오 Monte Carlo VaR (다변량)"""
    rng=np.random.default_rng(seed)
    ml=min(len(ret_dict[tk]) for tk in tickers)
    rm=np.column_stack([ret_dict[tk][-ml:] for tk in tickers])
    cov=ewma_cov_mat(rm,lam)*holding
    mu=rm.mean(axis=0)*holding
    w=np.array(weights)
    if use_t and df>2:
        L=np.linalg.cholesky(cov+np.eye(len(w))*1e-10)
        scale=np.sqrt((df-2)/df)
        z=stats.t.rvs(df=df,size=(n,len(w)),random_state=int(rng.integers(0,2**31)))
        sim_mat=mu+(z*scale)@L.T
    else:
        sim_mat=rng.multivariate_normal(mu,cov,n)
    pf_rets=sim_mat@w
    alpha=1-conf; vp=float(-np.percentile(pf_rets,alpha*100))
    cm=pf_rets<=-vp; cp=float(-pf_rets[cm].mean()) if cm.any() else vp
    ind_mc=[float(-np.percentile(sim_mat[:,i],alpha*100))*total_mv*w[i]
            for i in range(len(tickers))]
    return {"pf_var":vp*total_mv,"pf_cvar":cp*total_mv,"var_pct":vp,
            "ind_mc":ind_mc,"pf_rets":pf_rets}

dist_str = f"t-분포(ν={MC_DF})" if USE_T else "정규분포"
print(f"Monte Carlo 실행: {MC_N:,}회 | {dist_str} | {int(CONF*100)}% | {HOLDING}일")

ind_mcvars=[]; mc_rows=[]
for p,mv in zip(POSITIONS,mv_list):
    tk=p["ticker"]; r=RETURNS[tk]
    res=mc_var_individual(r,mv,HOLDING,CONF,MC_N,USE_T,MC_DF,LAM,MC_SEED)
    ind_mcvars.append(res["var"])
    mc_rows.append({"티커":tk,"MC-VaR(M)":f"{res['var']/1e6:.2f}",
                    "MC-CVaR(M)":f"{res['cvar']/1e6:.2f}","MC-VaR%":f"{res['var_pct']*100:.3f}%"})
print("\n=== 개별 종목 MC VaR ===")
print(pd.DataFrame(mc_rows).to_string(index=False))

pf_mc=mc_var_portfolio(RETURNS,tickers,weights,total_mv,HOLDING,CONF,MC_N,USE_T,MC_DF,LAM,MC_SEED)
print(f"\n=== 포트폴리오 MC VaR ===")
print(f"MC VaR  : {pf_mc['pf_var']/1e6:.2f}M원  ({pf_mc['var_pct']*100:.3f}%)")
print(f"MC CVaR : {pf_mc['pf_cvar']/1e6:.2f}M원")

print(f"\n=== 3종류 VaR 비교 ({int(CONF*100)}%, {HOLDING}일) ===")
cmp=pd.DataFrame({"방법":["Parametric (EWMA)","Historical Simulation",f"Monte Carlo ({dist_str})"],
    "포트폴리오 VaR(M)":[round(pf_res["pf_var"]/1e6,2),round(h_res["pf_var"]/1e6,2),round(pf_mc["pf_var"]/1e6,2)],
    "VaR%":[f"{pf_res['pf_var']/total_mv*100:.3f}%",f"{h_res['pf_var']/total_mv*100:.3f}%",f"{pf_mc['pf_var']/total_mv*100:.3f}%"]})
print(cmp.to_string(index=False))

### 5-1. Monte Carlo 손익 분포 및 3종류 VaR 비교

In [ ]:
if PLOTLY:
    # MC 손익 분포 히스토그램
    fig=go.Figure()
    fig.add_trace(go.Histogram(x=(pf_mc["pf_rets"]*total_mv/1e6).tolist(),
        nbinsx=60,marker_color="rgba(83,74,183,0.55)",name="MC 손익 분포"))
    fig.add_vline(x=-pf_mc["pf_var"]/1e6,line_dash="dash",line_color="#A32D2D",
                  annotation_text=f"MC VaR {int(CONF*100)}%")
    fig.add_vline(x=-pf_res["pf_var"]/1e6,line_dash="dot",line_color="#185FA5",
                  annotation_text=f"P-VaR {int(CONF*100)}%")
    fig.add_vline(x=-h_res["pf_var"]/1e6,line_dash="dot",line_color="#854F0B",
                  annotation_text=f"H-VaR {int(CONF*100)}%")
    fig.update_layout(title=f"포트폴리오 MC 손익 분포 ({dist_str}, {MC_N:,}회, 백만원)",
                      xaxis_title="손익 (백만원)",height=320,showlegend=False)
    fig.show()

    # 3종류 VaR 막대 비교
    methods=["Parametric","Historical",f"MC({dist_str})"]
    vals=[pf_res["pf_var"]/1e6,h_res["pf_var"]/1e6,pf_mc["pf_var"]/1e6]
    fig2=go.Figure(go.Bar(x=methods,y=vals,marker_color=["#A32D2D","#854F0B","#534AB7"],
        text=[f"{v:.1f}M" for v in vals],textposition="outside"))
    fig2.update_layout(title=f"3종류 VaR 비교 ({int(CONF*100)}%, {HOLDING}일, 백만원)",
                       height=300,showlegend=False)
    fig2.show()
else:
    plt.figure(figsize=(10,4))
    plt.hist(pf_mc["pf_rets"]*total_mv/1e6,bins=60,color="#534AB7",alpha=0.6)
    for v,c,lb in [(-pf_mc["pf_var"]/1e6,"r","MC VaR"),
                   (-pf_res["pf_var"]/1e6,"b","P-VaR"),
                   (-h_res["pf_var"]/1e6,"orange","H-VaR")]:
        plt.axvline(v,color=c,linestyle="--",label=lb)
    plt.title(f"MC 손익 분포 ({dist_str})"); plt.legend(); plt.tight_layout(); plt.show()

## 6. VaR Back-testing

**Basel 기준 (250영업일)**:
- 🟢 녹색: 0~4회 → 모형 적합, 자본승수 ×3.0
- 🟡 황색: 5~9회 → 경고, 자본승수 ×3.4~3.9  
- 🔴 적색: 10회+ → 모형 부적합, 자본승수 ×4.0

**Kupiec POF 검정**: $LR = 2[k\ln(p_e/p_0) + (n-k)\ln((1-p_e)/(1-p_0))] \sim \chi^2(1)$

In [ ]:
BT_TK=tickers[0]; BT_WIN=250; BT_CONF=0.99
n_total=BT_WIN+252
bt_ret,_,_=fetch_returns(BT_TK,n_total)
mv_bt=POSITIONS[0]["price"]*POSITIONS[0]["qty"]

# VaR 시계열 (1일 시차 적용)
pnl_s=bt_ret*mv_bt
vol_s=ewma_vol(pnl_s/mv_bt,LAM)*mv_bt
z_bt=stats.norm.ppf(BT_CONF)
var_s=z_bt*vol_s; var_s=np.roll(var_s,1); var_s[0]=var_s[1]

pnl_=pnl_s[-BT_WIN:]; var_=var_s[-BT_WIN:]
exc=int((-pnl_>var_).sum()); exc_r=exc/BT_WIN*100
zone=("🟢 녹색 (모형 적합)" if exc<=4 else
      "🟡 황색 (경고)" if exc<=9 else "🔴 적색 (부적합)")

from math import log as mlog
p_th=1-BT_CONF; k=exc; n=BT_WIN
lr_k=(2*(k*mlog(k/n/p_th+1e-12)+(n-k)*mlog((n-k)/n/(1-p_th)+1e-12))
      if k>0 else 0.0)
p_k=1-stats.chi2.cdf(lr_k,df=1)
kup="기각(부적합)" if p_k<0.05 else "채택(적합)"

print(f"=== VaR Back-testing: {BT_TK} ({int(BT_CONF*100)}%, {BT_WIN}일) ===")
print(f"초과 횟수 : {exc}일 ({exc_r:.2f}%)")
print(f"Basel 판정: {zone}")
print(f"Kupiec LR : {lr_k:.3f}  p={p_k:.4f}  → {kup}")

In [ ]:
# Back-testing 시각화
dates_bt=[(datetime.today()-timedelta(days=BT_WIN-i)).strftime("%m/%d")
          for i in range(BT_WIN)]
exc_idx=[i for i in range(BT_WIN) if -pnl_[i]>var_[i]]

if PLOTLY:
    fig=go.Figure()
    fig.add_trace(go.Bar(x=dates_bt,y=pnl_.tolist(),name="실제 손익",
        marker_color=["rgba(162,45,45,0.7)" if v<0 else "rgba(55,138,221,0.5)" for v in pnl_]))
    fig.add_trace(go.Scatter(x=dates_bt,y=(-var_).tolist(),mode="lines",
        name=f"VaR {int(BT_CONF*100)}%",line=dict(color="#A32D2D",dash="dash",width=1.8)))
    if exc_idx:
        fig.add_trace(go.Scatter(x=[dates_bt[i] for i in exc_idx],
            y=[float(pnl_[i]) for i in exc_idx],mode="markers",
            name=f"VaR 초과({len(exc_idx)}일)",marker=dict(color="#A32D2D",size=9,symbol="x")))
    fig.update_layout(title=f"Back-testing: {BT_TK} ({int(BT_CONF*100)}%, {BT_WIN}일)",
                      height=380,legend=dict(orientation="h",y=-0.2))
    fig.show()
else:
    plt.figure(figsize=(14,5))
    plt.bar(range(BT_WIN),pnl_,color=["r" if v<0 else "b" for v in pnl_],alpha=0.6)
    plt.plot(range(BT_WIN),-var_,"r--",lw=1.5,label=f"VaR {int(BT_CONF*100)}%")
    for i in exc_idx: plt.scatter(i,pnl_[i],color="red",s=60,zorder=5)
    plt.title(f"Back-testing: {BT_TK}"); plt.legend(); plt.tight_layout(); plt.show()

## 7. 리포트 생성 (Excel + Markdown)

In [ ]:
import io
try:
    from openpyxl import Workbook
    from openpyxl.styles import Font,PatternFill,Alignment,Border,Side
    from openpyxl.utils import get_column_letter; OPX=True
except: OPX=False; print("openpyxl 없음: pip install openpyxl")

def fmt_n(v):
    a=abs(v)
    if a>=1e12: return f"{v/1e12:.2f}조"
    if a>=1e8:  return f"{v/1e8:.2f}억"
    if a>=1e4:  return f"{v/1e4:,.0f}만"
    return f"{v:,.0f}"

now=datetime.now().strftime("%Y-%m-%d %H:%M:%S")
ts =datetime.now().strftime("%Y%m%d_%H%M")

if OPX:
    wb=Workbook(); wb.remove(wb.active)
    thin=Side(style="thin",color="BFBFBF")
    bdr=Border(left=thin,right=thin,top=thin,bottom=thin)
    def hd(ws,r,cols,texts,bg="1F3864",fg="FFFFFF"):
        for c,t in zip(cols,texts):
            cell=ws.cell(row=r,column=c,value=t)
            cell.font=Font(bold=True,color=fg,size=10,name="Arial")
            cell.fill=PatternFill("solid",start_color=bg)
            cell.alignment=Alignment(horizontal="center",vertical="center")
            cell.border=bdr
    def cl(ws,r,c,v,bold=False,fmt=None,fg="000000",bg=None,align="right"):
        cell=ws.cell(row=r,column=c,value=v)
        cell.font=Font(bold=bold,color=fg,name="Arial",size=10)
        cell.alignment=Alignment(horizontal=align,vertical="center")
        cell.border=bdr
        if bg: cell.fill=PatternFill("solid",start_color=bg)
        if fmt: cell.number_format=fmt

    # Sheet 1: 포트폴리오
    ws1=wb.create_sheet("포트폴리오"); ws1.sheet_view.showGridLines=False
    ws1.column_dimensions["A"].width=22
    for c2,w2 in zip("BCDEFG",[10,12,10,18,10,10]): ws1.column_dimensions[c2].width=w2
    ws1.merge_cells("A1:G1"); ws1["A1"]=f"포트폴리오 구성 | {now}"
    ws1["A1"].font=Font(bold=True,color="FFFFFF",size=13,name="Arial")
    ws1["A1"].fill=PatternFill("solid",start_color="1F3864")
    ws1["A1"].alignment=Alignment(horizontal="center",vertical="center")
    hd(ws1,2,range(1,8),["티커","유형","시장","수량","평가금액","통화","비중"],bg="2F5496")
    for i,p in enumerate(POSITIONS):
        mv=p["price"]*p["qty"]*(USDKRW if p["ccy"]=="USD" else 1)*p.get("mult",1)
        bg="F2F2F2" if i%2==0 else "FFFFFF"; r=i+3
        cl(ws1,r,1,p["ticker"],bold=True,align="left",bg=bg)
        cl(ws1,r,2,p["type"],align="center",bg=bg)
        cl(ws1,r,3,p.get("market",""),align="center",bg=bg)
        cl(ws1,r,4,p["qty"],fmt="#,##0",bg=bg)
        cl(ws1,r,5,mv,fmt="#,##0",bg=bg)
        cl(ws1,r,6,p["ccy"],align="center",bg=bg)
        cl(ws1,r,7,mv/total_mv,fmt="0.00%",bg=bg)

    # Sheet 2: VaR 비교
    ws2=wb.create_sheet("VaR 비교"); ws2.sheet_view.showGridLines=False
    ws2.merge_cells("A1:F1"); ws2["A1"]=f"Parametric / Historical / MC VaR | {int(CONF*100)}% | {HOLDING}일"
    ws2["A1"].font=Font(bold=True,color="FFFFFF",size=13,name="Arial")
    ws2["A1"].fill=PatternFill("solid",start_color="1F3864")
    ws2["A1"].alignment=Alignment(horizontal="center",vertical="center")
    for ci3,w3 in enumerate([22,16,14,14,14,14],1): ws2.column_dimensions[get_column_letter(ci3)].width=w3
    hd(ws2,2,range(1,7),["방법","포트폴리오 VaR","포트폴리오 CVaR","VaR%","비고",""],bg="2F5496")
    var_data=[("Parametric (EWMA)",pf_res["pf_var"],pf_res["pf_var"]*1.05,pf_res["pf_var"]/total_mv,"EWMA λ=0.94"),
              ("Historical Simulation",h_res["pf_var"],h_res["pf_cvar"],h_res["pf_var"]/total_mv,"분포 가정 없음"),
              (f"Monte Carlo ({dist_str})",pf_mc["pf_var"],pf_mc["pf_cvar"],pf_mc["pf_var"]/total_mv,f"t(ν={MC_DF}), {MC_N:,}회")]
    for i,(nm,v,cv,vp,note) in enumerate(var_data):
        bg="F2F2F2" if i%2==0 else "FFFFFF"
        cl(ws2,3+i,1,nm,bold=True,align="left",bg=bg)
        cl(ws2,3+i,2,v,fmt="#,##0",fg="C00000",bg=bg)
        cl(ws2,3+i,3,cv,fmt="#,##0",fg="7F6000",bg=bg)
        cl(ws2,3+i,4,vp,fmt="0.000%",bg=bg)
        cl(ws2,3+i,5,note,align="left",bg=bg)

    # Sheet 3: 종목별 VaR
    ws3=wb.create_sheet("종목별 VaR"); ws3.sheet_view.showGridLines=False
    for ci4,w4 in enumerate([20,14,14,12,14,12,14,12,14,10,10],1):
        ws3.column_dimensions[get_column_letter(ci4)].width=w4
    ws3.merge_cells("A1:K1"); ws3["A1"]=f"종목별 VaR 상세 | {int(CONF*100)}% | {HOLDING}일"
    ws3["A1"].font=Font(bold=True,color="FFFFFF",size=13,name="Arial")
    ws3["A1"].fill=PatternFill("solid",start_color="1F3864")
    ws3["A1"].alignment=Alignment(horizontal="center",vertical="center")
    hd(ws3,2,range(1,12),["티커","P-VaR","P-CVaR","P-VaR%","H-VaR","H-VaR%",
                            "MC-VaR","Component VaR","기여도","IVaR","Beta"],bg="2F5496")
    for i,tk in enumerate(tickers):
        bg="F2F2F2" if i%2==0 else "FFFFFF"; r=i+3
        pv_=ind_pvars[i]; hv_=ind_hvars[i]; mc_=ind_mcvars[i]
        comp_=pf_res["comp"][i]; ct=comp_/pf_res["pf_var"]*100 if pf_res["pf_var"] else 0
        cl(ws3,r,1,tk,bold=True,align="left",bg=bg)
        cl(ws3,r,2,pv_,fmt="#,##0",fg="C00000",bg=bg)
        cl(ws3,r,3,pv_*1.05,fmt="#,##0",fg="7F6000",bg=bg)
        cl(ws3,r,4,pv_/mv_list[i],fmt="0.000%",bg=bg)
        cl(ws3,r,5,hv_,fmt="#,##0",fg="C00000",bg=bg)
        cl(ws3,r,6,hv_/mv_list[i],fmt="0.000%",bg=bg)
        cl(ws3,r,7,mc_,fmt="#,##0",fg="C00000",bg=bg)
        cl(ws3,r,8,comp_,fmt="#,##0",bg=bg)
        cl(ws3,r,9,ct/100,fmt="0.0%",bg=bg)
        cl(ws3,r,10,ivars[i],fmt="#,##0",bg=bg)
        cl(ws3,r,11,pf_res["beta"][i],fmt="0.000",bg=bg)

    # Sheet 4: Back-testing
    ws4=wb.create_sheet("Back-testing"); ws4.sheet_view.showGridLines=False
    for ci5,w5 in enumerate([26,20],1): ws4.column_dimensions[get_column_letter(ci5)].width=w5
    ws4.merge_cells("A1:B1"); ws4["A1"]=f"VaR Back-testing | {BT_TK} | {int(BT_CONF*100)}% | {BT_WIN}일"
    ws4["A1"].font=Font(bold=True,color="FFFFFF",size=13,name="Arial")
    ws4["A1"].fill=PatternFill("solid",start_color="1F3864")
    ws4["A1"].alignment=Alignment(horizontal="center",vertical="center")
    hd(ws4,2,[1,2],["지표","값"],bg="2F5496")
    bt_d=[("VaR 초과 횟수",exc),("초과 비율(%)",round(exc_r,2)),("Basel 판정",zone),
          ("Kupiec LR",round(lr_k,3)),("Kupiec p-value",round(p_k,4)),("Kupiec 결과",kup)]
    for i,(k2,v2) in enumerate(bt_d):
        bg="F2F2F2" if i%2==0 else "FFFFFF"
        fg=("C00000" if exc>=10 else "7F6000" if exc>=5 else "375623") if k2=="VaR 초과 횟수" else "000000"
        cl(ws4,3+i,1,k2,align="left",bg=bg); cl(ws4,3+i,2,str(v2) if isinstance(v2,str) else v2,fg=fg,bg=bg)

    fname_xl=f"VaR_Report_{ts}.xlsx"; wb.save(fname_xl)
    print(f"Excel 저장: {fname_xl}  ({[ws.title for ws in wb.worksheets]})")

In [ ]:
# Markdown 리포트
L=[]; a=L.append
a("# 포트폴리오 리스크 분석 리포트"); a()
a(f"| 작성일시 | {now} | 신뢰수준 | {int(CONF*100)}% | 보유기간 | {HOLDING}일 |")
a("|------|------|------|------|------|------|"); a(); a("---"); a()
a("## 1. 포트폴리오 구성"); a()
a(f"- 총 평가금액: {fmt_n(total_mv)}원  |  종목 수: {len(POSITIONS)}개"); a()
a("| 티커 | 유형 | 수량 | 평가금액 | 비중 |")
a("|------|------|-----:|--------:|-----:|")
for p,mv in zip(POSITIONS,mv_list):
    a(f"| {p['ticker']} | {p['type']} | {p['qty']:,.0f} | {fmt_n(mv)}원 | {mv/total_mv*100:.1f}% |")
a(); a("---"); a("## 2. VaR 비교 (3종류)"); a()
a(f"| 방법 | 포트폴리오 VaR | VaR% |")
a("|------|-------------:|-----:|")
a(f"| Parametric (EWMA) | {fmt_n(pf_res['pf_var'])}원 | {pf_res['pf_var']/total_mv*100:.3f}% |")
a(f"| Historical Simulation | {fmt_n(h_res['pf_var'])}원 | {h_res['pf_var']/total_mv*100:.3f}% |")
a(f"| Monte Carlo ({dist_str}) | {fmt_n(pf_mc['pf_var'])}원 | {pf_mc['pf_var']/total_mv*100:.3f}% |")
a(); a("---"); a("## 3. 리스크 기여도 (Component VaR)"); a()
a("| 티커 | 비중 | P-VaR | H-VaR | MC-VaR | CVaR | 기여도 | IVaR | Beta |")
a("|------|-----:|------:|------:|-------:|-----:|-------:|-----:|-----:|")
for i,tk in enumerate(tickers):
    ct=pf_res["comp"][i]/pf_res["pf_var"]*100 if pf_res["pf_var"] else 0
    a(f"| {tk} | {weights[i]*100:.1f}% | {ind_pvars[i]/1e6:.1f}M | "
      f"{ind_hvars[i]/1e6:.1f}M | {ind_mcvars[i]/1e6:.1f}M | "
      f"{pf_res['comp'][i]/1e6:.1f}M | {ct:.1f}% | {ivars[i]/1e6:.1f}M | {pf_res['beta'][i]:.3f} |")
a(); a("---"); a("## 4. Back-testing"); a()
a(f"- 종목: {BT_TK}  |  초과: {exc}일 ({exc_r:.2f}%)  |  {zone}")
a(f"- Kupiec LR={lr_k:.3f}, p={p_k:.4f} → {kup}")
a(); a("---"); a(f"*VaR_Cal.ipynb 자동 생성 | {now}*")

md_text="\n".join(L)
fname_md=f"VaR_Report_{ts}.md"
with open(fname_md,"w",encoding="utf-8") as f: f.write(md_text)
print(f"MD 저장: {fname_md}  ({len(L)}줄)")

## 8. 기법 선택 가이드

| 기법 | 핵심 강점 | 적합 자산 | 주요 한계 |
|------|----------|----------|----------|
| **Parametric (EWMA)** | 변동성 클러스터링 반영, 바젤 직접 활용 | 주식·채권 포트폴리오 | 정규분포 가정 |
| **Historical Simulation** | 분포 가정 없음, Fat-tail 자동 반영 | 비선형 자산, 실제 분포 이상 시 | 표본 외 사건 반영 불가 |
| **Monte Carlo (t-분포)** | Fat-tail 반영, 극단 시나리오 포착 | 포트폴리오 전체 리스크 관리 | 자유도 선택 민감도 |
| **Merton JD** | 불연속 점프 충격 포착 | 개별 주식, 옵션 | 파라미터 추정 어려움 |
| **Block Bootstrap** | 실제 패턴 재현, 자기상관 보존 | 지수·ETF | 블록 크기 민감도 |

> 실행: `streamlit run VaR_Cal.py`